# Multicollinearity

**Multicollinearity** occurs when two or more predictors in a regression model are highly correlated with each other. This doesn’t stop the model from fitting the data well — but it makes the individual coefficient estimates unreliable. When predictors share information, the model can’t confidently say how much of the response each one is responsible for.

In this notebook we’ll:
- Demonstrate the problem using a synthetic example where we know the ground truth
- Build intuition for *why* correlated predictors inflate standard errors
- Use an interactive visualization to see the effect as correlation increases
- Detect multicollinearity in real data using a **Variance Inflation Factor (VIF)**

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

import ipywidgets as widgets
from ipywidgets import interact

import warnings
warnings.filterwarnings('ignore')

plt.rc('figure', figsize=(12, 5))
np.set_printoptions(precision=4, suppress=True)

## Part 1: A Synthetic Example

We’ll generate data where we know exactly what the true relationship is. That way we can see clearly when the estimates go wrong.

The true model is:

$$y = 2 + 2x_1 + 0.3x_2 + \varepsilon$$

So the true coefficients are $\beta_0 = 2$, $\beta_1 = 2$, $\beta_2 = 0.3$. We’ll make $x_2$ a noisy function of $x_1$ — the two predictors will be highly correlated.

In [ ]:
np.random.seed(0)
x1 = np.random.uniform(size=100)
x2 = 0.5 * x1 + np.random.normal(size=100) / 10
y  = 2 + 2 * x1 + 0.3 * x2 + np.random.normal(size=100)

print(f'Correlation between x1 and x2: {np.corrcoef(x1, x2)[0, 1]:.4f}')

The correlation is around 0.84 — high, but not perfect. Let’s visualize the relationship.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x1, x2, alpha=0.6, color='steelblue')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('x1 vs x2 — correlated predictors')
plt.tight_layout()
plt.show()

### Fitting three models

Now we’ll fit three regressions and compare the coefficient estimates and p-values:
1. Both predictors together
2. Only $x_1$
3. Only $x_2$

In [ ]:
df = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y})

model_both = smf.ols('y ~ x1 + x2', df).fit()
model_x1   = smf.ols('y ~ x1',      df).fit()
model_x2   = smf.ols('y ~ x2',      df).fit()

print('=== Both predictors ===')
print(model_both.summary().tables[1])

print('\n=== x1 only ===')
print(model_x1.summary().tables[1])

print('\n=== x2 only ===')
print(model_x2.summary().tables[1])

### Reading the results

Look at the p-values carefully:

| Model | $x_1$ significant? | $x_2$ significant? |
|---|---|---|
| Both predictors | Yes | **No** |
| $x_1$ only | Yes | — |
| $x_2$ only | — | **Yes** |

This is the multicollinearity paradox: $x_2$ is clearly significant when it’s the only predictor, but becomes non-significant when $x_1$ is also in the model. Did $x_2$ suddenly stop mattering?

No. What happened is that $x_1$ already contains most of the same information as $x_2$. Once the model knows $x_1$, adding $x_2$ tells it almost nothing new. The model can’t reliably separate their individual contributions, so it gives both wide confidence intervals and high p-values.

Notice also that the coefficient estimate for $x_1$ in the joint model is close to the true value of 2, but the standard error is much larger than when $x_1$ is alone. The estimate is still unbiased — but it’s *noisy*.

## Part 2: Why It Happens — Standard Error Inflation

In ordinary least squares, the variance of a coefficient estimate $\hat{\beta}_j$ is:

$$\text{Var}(\hat{\beta}_j) = \frac{\sigma^2}{\text{SS}_{x_j}(1 - R_j^2)}$$

where $R_j^2$ is the R-squared from regressing $x_j$ on all the other predictors.

When $x_j$ is highly correlated with another predictor, $R_j^2 \to 1$, so $(1 - R_j^2) \to 0$, and the variance of $\hat{\beta}_j$ blows up. This is exactly what we saw: the standard errors for both coefficients in the joint model are much larger than the standard errors in the single-predictor models.

## Part 3: Watching It Happen — Varying Correlation

The widget below generates data with a tunable correlation between $x_1$ and $x_2$. As you increase the correlation, watch what happens to:
- The standard errors of the coefficient estimates (shown in parentheses)
- The p-values
- The confidence intervals (shown as error bars)

In [ ]:
@interact(rho=widgets.FloatSlider(min=0.0, max=0.99, step=0.05, value=0.0,
                                   description='Correlation ρ',
                                   style={'description_width': 'initial'}))
def show_multicollinearity(rho):
    np.random.seed(42)
    n = 200
    x1_ = np.random.normal(size=n)
    x2_ = rho * x1_ + np.sqrt(max(1 - rho**2, 1e-10)) * np.random.normal(size=n)
    y_  = 2 + 2 * x1_ + 0.3 * x2_ + np.random.normal(size=n)

    df_ = pd.DataFrame({'x1': x1_, 'x2': x2_, 'y': y_})
    m = smf.ols('y ~ x1 + x2', df_).fit()

    coefs  = m.params[['x1', 'x2']]
    errors = m.bse[['x1', 'x2']]
    pvals  = m.pvalues[['x1', 'x2']]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = ['steelblue', 'darkorange']
    axes[0].barh(['x1', 'x2'], coefs, xerr=errors, color=colors, alpha=0.7, capsize=5)
    axes[0].axvline(2.0, color='steelblue', linestyle='--', linewidth=1.5, label='True β₁ = 2.0')
    axes[0].axvline(0.3, color='darkorange', linestyle='--', linewidth=1.5, label='True β₂ = 0.3')
    axes[0].axvline(0, color='gray', linewidth=0.8)
    axes[0].set_xlabel('Coefficient estimate ± 1 SE')
    axes[0].set_title(f'Coefficients (ρ = {rho:.2f})')
    axes[0].legend(fontsize=9)

    axes[1].bar(['x1 SE', 'x2 SE'], errors, color=colors, alpha=0.7)
    axes[1].set_ylabel('Standard error')
    axes[1].set_title('Standard errors grow as correlation increases')
    for i, (se, pv) in enumerate(zip(errors, pvals)):
        sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else 'n.s.'
        axes[1].text(i, se + 0.01, f'p={pv:.3f} {sig}', ha='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    print(f'x1 SE: {errors["x1"]:.4f}   x2 SE: {errors["x2"]:.4f}')
    print(f'x1 p:  {pvals["x1"]:.4f}   x2 p:  {pvals["x2"]:.4f}')

## Part 4: Detecting Multicollinearity in Real Data

In real datasets you rarely know the correlation structure in advance. Two tools help:

**1. Correlation matrix** — a quick visual scan for pairwise correlations. Anything above ~0.7 or 0.8 in absolute value is worth investigating.

**2. Variance Inflation Factor (VIF)** — a more rigorous measure. The VIF for predictor $x_j$ is:

$$\text{VIF}(\hat{\beta}_j) = \frac{1}{1 - R_j^2}$$

where $R_j^2$ comes from regressing $x_j$ on all other predictors. Interpreting VIF:
- VIF = 1: no collinearity with other predictors
- VIF 1–10: moderate collinearity, usually acceptable
- VIF > 10: high collinearity, coefficient estimates may be unreliable

Let’s apply both to the Hitters baseball dataset — 19 predictors of player salary.

In [ ]:
url = 'https://raw.githubusercontent.com/dsahota-applied-data-analysis/data/main/Hitters.csv'
hitters = pd.read_csv(url, index_col=0).dropna()

dummies = pd.get_dummies(hitters[['League', 'Division', 'NewLeague']], drop_first=True)
X = hitters.drop(['Salary', 'League', 'Division', 'NewLeague'], axis=1).astype('float64')
X = pd.concat([X, dummies], axis=1).astype('float64')

print(f'{len(hitters)} observations, {X.shape[1]} predictors')
X.head()

In [ ]:
# Correlation matrix heatmap
numeric_cols = hitters.drop(['Salary', 'League', 'Division', 'NewLeague'], axis=1).columns

fig, ax = plt.subplots(figsize=(12, 10))
corr = hitters[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, ax=ax, annot_kws={'size': 8})
ax.set_title('Pairwise correlations among numeric predictors in Hitters', fontsize=13)
plt.tight_layout()
plt.show()

Several predictors are strongly correlated — career stats (`CAtBat`, `CHits`, `CHmRun`, `CRuns`, `CRBI`, `CWalks`) are all highly correlated with each other, which makes sense: players who have played longer accumulate more of everything.

Now let’s compute the VIF for each predictor.

In [ ]:
vif_df = pd.DataFrame({
    'Feature': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

vif_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['crimson' if v > 10 else 'darkorange' if v > 5 else 'steelblue'
          for v in vif_df['VIF']]
ax.barh(vif_df['Feature'], vif_df['VIF'], color=colors)
ax.axvline(10, color='crimson', linestyle='--', linewidth=1.5, label='VIF = 10 (high collinearity)')
ax.axvline(5, color='darkorange', linestyle='--', linewidth=1.5, label='VIF = 5 (moderate)')
ax.set_xlabel('Variance Inflation Factor')
ax.set_title('VIF by predictor — Hitters dataset')
ax.legend()
plt.tight_layout()
plt.show()

The career counting stats (`CAtBat`, `CHits`, etc.) have very high VIFs — exactly what we’d expect given the high pairwise correlations. Their coefficient estimates in a full model will have inflated standard errors and may appear non-significant even if they genuinely predict salary.

## Part 5: What To Do About Multicollinearity

Several options, depending on the goal:

**Drop one of the correlated predictors** — if two predictors carry nearly the same information, keeping both buys you little. Drop the one that’s less interpretable or less directly relevant.

**Combine correlated predictors** — for the career stats, you might create a single “career productivity” feature rather than including all of them separately.

**Use Ridge regression** — Ridge explicitly penalizes large coefficients, which stabilizes estimates when predictors are collinear. This is covered in notebook 15. Ridge doesn’t fix the underlying problem, but it produces more stable (if biased) estimates.

**Collect more data** — multicollinearity is partly a data problem. With more observations, the model has more information to disentangle the predictors’ individual contributions.

Note that multicollinearity does *not* degrade a model’s predictive accuracy in-sample — predictions from the joint model are fine. The problem is purely in interpreting the individual coefficients. If prediction is all you care about, multicollinearity is less of a concern.